# Session 1: charges, fields and waves
**A. Real-world problem.** Rub a balloon on your hair and it sticks to the wall; the same
invisible push that does this also carries your FM radio and wifi through the air. What is
that push, what does it look like, and why do light and radio turn out to be the same thing?

**Plan.** (1) charges push or pull, (2) draw the push as arrows (the *electric field*) for
like charges and for a dipole, (3) a wave is a disturbance that travels, (4) speed =
frequency x wavelength, (5) light and radio travel at the same speed.

In [ ]:
import pathlib

import numpy as np

OUT = (pathlib.Path(__file__).resolve().parent if "__file__" in globals() else pathlib.Path(".")) / "out"
C = 299792458.0            # speed of light in vacuum, metres per second (about 3e8)

## Step 1: like charges repel, opposite charges attract
A charge is positive or negative. Two charges of the *same* sign push apart; *opposite* signs
pull together. Write a charge as a signed number: +1, -1, +2 ...

In [ ]:
def force_direction(q1, q2):
    """'repel', 'attract' or 'none' for two charges (signed numbers)."""
    if q1 == 0 or q2 == 0:
        return "none"
    return "repel" if q1 * q2 > 0 else "attract"

## Step 2: the field is "the push a tiny test charge would feel"
Put a tiny positive test charge at a point. Each charge pushes it: away from positive charges,
towards negative ones, and the push is weaker far away. Add up the pushes as arrows. The
result is the *electric field* at that point. Here we draw it in simple units (the number
is the strength of a push, the exact scale comes in Session 2).

Two equal positive charges: the pushes cancel exactly in the middle. A dipole (one positive,
one negative of equal size): in the middle both pushes point the same way, from + towards -.

In [ ]:
def field_at(point, charges):
    """Field vector (Ex, Ey) at `point` in simple units. charges = [(q, x, y), ...].
    Each charge adds q * (point - charge position) / distance^3."""
    ex = ey = 0.0
    for q, x, y in charges:
        dx, dy = point[0] - x, point[1] - y
        r = np.hypot(dx, dy)
        if r == 0:
            continue
        ex += q * dx / r ** 3
        ey += q * dy / r ** 3
    return np.array([ex, ey])


def field_grid(charges, xs, ys):
    """Field on a grid. Returns Ex, Ey arrays of shape (len(ys), len(xs))."""
    X, Y = np.meshgrid(xs, ys)
    Ex = np.zeros_like(X)
    Ey = np.zeros_like(X)
    for q, x, y in charges:
        dx, dy = X - x, Y - y
        r3 = np.maximum(np.hypot(dx, dy), 1e-9) ** 3
        Ex += q * dx / r3
        Ey += q * dy / r3
    return Ex, Ey


LIKE = [(1.0, -1.0, 0.0), (1.0, 1.0, 0.0)]          # two positive charges
DIPOLE = [(1.0, -1.0, 0.0), (-1.0, 1.0, 0.0)]       # one positive, one negative

## Step 3: a wave is a disturbance that travels
Shake the end of a rope: a bump runs along it. The rope itself only moves up and down; what
*travels* is the bump. In radio and light, what travels is a changing electric and magnetic
field. A bump at position x and time t, moving right at speed v, is simply a bump that was
at x - v t at time 0.

In [ ]:
def pulse(x, t, v, width=1.0):
    """A bump (height 1) that started at x = 0 and moves right at speed v."""
    return np.exp(-(((np.asarray(x) - v * t) / width) ** 2))


def peak_position(x, t, v, width=1.0):
    """Where the highest point of the bump is at time t (read from the sampled curve)."""
    y = pulse(x, t, v, width)
    return float(np.asarray(x)[np.argmax(y)])

## Step 4: speed = frequency x wavelength
A repeating wave has a *wavelength* (distance between neighbouring crests, in metres) and a
*frequency* (crests passing a point per second, in hertz). If f crests pass each second and
each is lambda metres long, the wave moves f * lambda metres each second:  v = f * lambda.

In [ ]:
def wavelength(f, v=C):
    """Wavelength in metres for frequency f (Hz) and wave speed v (m/s)."""
    return v / f


def frequency(lam, v=C):
    """Frequency in Hz for wavelength lam (m)."""
    return v / lam

## Step 5: light and radio are the same kind of wave
In empty space every electromagnetic wave travels at c, about 3e8 m/s (300 million metres per
second), whatever its frequency. Radio, wifi, visible light and X-rays differ only in
frequency and wavelength.

In [ ]:
EXAMPLES = {
    "FM radio, 100 MHz": 100e6,
    "wifi, 2.4 GHz": 2.4e9,
    "green light, 600 THz": 600e12,
}

In [ ]:
if __name__ == "__main__":
    import matplotlib.pyplot as plt

    OUT.mkdir(exist_ok=True)
    print("two positive charges:", force_direction(1, 1), "| a positive and a negative charge:", force_direction(1, -1))
    print("field in the middle, like charges :", field_at((0.0, 0.0), LIKE).round(3))
    print("field in the middle, dipole       :", field_at((0.0, 0.0), DIPOLE).round(3), "(points from + towards -)")
    for name, f in EXAMPLES.items():
        print(f"{name}: wavelength = {wavelength(f):.4g} m")
    print(f"a sound wave at 440 Hz in air (343 m/s): wavelength = {wavelength(440, 343.0):.3f} m")
    x = np.linspace(-2, 12, 1401)
    for t in (0.0, 2.0, 5.0):
        print(f"pulse at t = {t}: peak at x = {peak_position(x, t, 2.0):.2f} (v t = {2.0 * t:.1f})")

    xs = np.linspace(-3, 3, 121)
    fig, axs = plt.subplots(1, 2, figsize=(9, 4.2))
    for ax, ch, title in ((axs[0], LIKE, "Two like charges: they repel"), (axs[1], DIPOLE, "A dipole: lines run from + to -")):
        Ex, Ey = field_grid(ch, xs, xs)
        mag = np.hypot(Ex, Ey)
        ax.streamplot(xs, xs, Ex, Ey, color=np.log10(mag + 1e-3), cmap="viridis", density=1.4, linewidth=0.9)
        for q, cx, cy in ch:
            ax.plot(cx, cy, "o", ms=14, color="tab:red" if q > 0 else "tab:blue")
            ax.text(cx, cy, "+" if q > 0 else "-", ha="center", va="center", color="white", fontsize=12)
        ax.set_xlim(-3, 3)
        ax.set_ylim(-3, 3)
        ax.set_aspect("equal")
        ax.set_xlabel("x (arbitrary length unit)")
        ax.set_ylabel("y (arbitrary length unit)")
        ax.set_title(title)
    fig.tight_layout()
    fig.savefig(OUT / "field_lines.png", dpi=110)

    fig2, axs2 = plt.subplots(1, 2, figsize=(10, 4))
    for t, col in ((0.0, "tab:blue"), (2.0, "tab:orange"), (5.0, "tab:green")):
        axs2[0].plot(x, pulse(x, t, 2.0), color=col, label=f"t = {t:.0f} s")
    axs2[0].set_xlabel("position x (m)")
    axs2[0].set_ylabel("disturbance (relative)")
    axs2[0].set_title("A bump travelling at v = 2 m/s")
    axs2[0].legend()
    axs2[0].grid(alpha=0.3)
    f = np.logspace(5, 16, 200)
    axs2[1].loglog(f, wavelength(f), color="black")
    for name, fv in EXAMPLES.items():
        axs2[1].plot(fv, wavelength(fv), "o", label=name)
    axs2[1].set_xlabel("frequency (Hz)")
    axs2[1].set_ylabel("wavelength (m)")
    axs2[1].set_title("lambda = c / f: one line for all light and radio")
    axs2[1].legend(fontsize=7)
    axs2[1].grid(alpha=0.3, which="both")
    fig2.tight_layout()
    fig2.savefig(OUT / "wave_basics.png", dpi=110)
    print("saved figures in", OUT)

## Try it
Do the coding tasks, answer the four questions in the README (section F), then run the
checkpoint.

## Coding tasks
1. **Easy.** Change `LIKE` in `lesson.py` so the two charges are +1 and +2. Predict where the field is zero (closer to the weaker charge), then check with `field_at`.
2. **Stretch.** Write `colour_wavelength(f)` for red light (450 THz) and blue light (650 THz) and print both in nanometres. Which is longer?
3. **Explain.** Explain why a sound wave of 440 Hz has a wavelength of about 0.78 m, much shorter than the 680 km a radio wave of the same frequency would have. (Hint: compare the two speeds.)

## Assessment
- **A. Conceptual.** Two protons and two electrons are placed in a line. Explain, without calculating, which pairs push apart and which pull together. Why does the field of two equal positive charges vanish halfway between them?
- **B. Numerical.** A radio wave has frequency 90 MHz. Find its wavelength. Check that the answer is sensible by comparing with the 100 MHz result.
- **C. Graph and simulation interpretation.** In the lambda-versus-frequency graph, FM radio, wifi and green light lie on one straight line. What does that tell you about their speeds? If you moved a point above the line, what would it mean?
- **D. Design and optimisation challenge.** Design: you want a wave whose wavelength is 1 m so that it fits a room-sized test. What frequency do you need? Is that a radio or a light wave? Would a 1 m wave pass through a 5 cm gap in a wall as easily as a 12.5 cm wifi wave?

## Checkpoint
Run this last cell. It prints PASS when your work is correct.

In [ ]:
import numpy as np


def run_checks(L):
    """L is the lesson module (or notebook namespace). Returns a list of (label, passed)."""
    checks = []
    checks.append(("like charges repel", L.force_direction(1, 2) == "repel" and L.force_direction(-1, -3) == "repel"))
    checks.append(("opposite charges attract", L.force_direction(1, -1) == "attract"))
    checks.append(("a zero charge feels nothing (edge case)", L.force_direction(0, 5) == "none"))

    mid_like = L.field_at((0.0, 0.0), L.LIKE)
    checks.append(("two equal positive charges: the field cancels in the middle", np.allclose(mid_like, 0)))
    mid_dip = L.field_at((0.0, 0.0), L.DIPOLE)
    checks.append(("dipole: the field in the middle points from + towards - and is 2", np.allclose(mid_dip, [2.0, 0.0])))
    checks.append(("a point exactly on a charge does not crash", np.all(np.isfinite(L.field_at((-1.0, 0.0), L.DIPOLE)))))
    Ex, Ey = L.field_grid(L.DIPOLE, np.linspace(-3, 3, 7), np.linspace(-3, 3, 7))
    checks.append(("field grid is finite everywhere", bool(np.all(np.isfinite(Ex)) and np.all(np.isfinite(Ey)))))
    far = L.field_at((50.0, 0.0), L.LIKE)
    checks.append(("far from two like charges the field is about that of one charge of size 2 (2/r^2)",
                   abs(far[0] - 2 / 50 ** 2) / (2 / 50 ** 2) < 0.01))

    # worked example: FM radio at 100 MHz
    lam = L.wavelength(100e6)
    checks.append((f"worked example: an FM station at 100 MHz has wavelength about 3.0 m (got {lam:.3f})", abs(lam - 2.998) < 0.001))
    checks.append(("frequency and wavelength are inverse of each other", abs(L.frequency(L.wavelength(5e8)) - 5e8) < 1))
    checks.append(("hand calculation: 2.4 GHz wifi wavelength is 0.125 m", abs(L.wavelength(2.4e9) - 0.125) < 0.001))
    checks.append(("green light 600 THz is about 500 nm", abs(L.wavelength(600e12) - 500e-9) < 2e-9))
    checks.append(("sound is much slower: 440 Hz in air gives 0.78 m", abs(L.wavelength(440, 343.0) - 0.78) < 0.001))

    x = np.linspace(-2, 12, 1401)
    checks.append(("the bump's peak moves at v: x = 10 at t = 5 when v = 2", abs(L.peak_position(x, 5.0, 2.0) - 10.0) < 0.02))
    checks.append(("at t = 0 the bump is at x = 0", abs(L.peak_position(x, 0.0, 2.0)) < 0.02))
    return checks

import types
_ns = types.SimpleNamespace(**{k: v for k, v in globals().items() if not k.startswith('_')})
_ok = True
for _label, _passed in run_checks(_ns):
    print(('PASS  ' if _passed else 'FAIL  ') + _label)
    _ok &= bool(_passed)
print('PASS' if _ok else 'FAIL')
